# TCGA expression, staging, and lncRNA tables

End-to-end pipeline: load Xena TCGA gene expected counts, keep primary tumor aliquots (`-01`), reshape to samples × genes, merge survival/clinical staging from Xena and harmonized AJCC fields from GDC, export stage-filtered and metastasis-stratified matrices, then build a GENCODE v23 lncRNA gene list and write lncRNA-only expression slices.


## 1. Imports


In [1]:
import re

import numpy as np
import pandas as pd


## 2. Load raw inputs

- **Expression:** `data/tcga_gene_expected_count.gz` (Xena); first column is probe/Ensembl id mapped to `gene_symbol` via probemap.
- **Xena clinical:** `data/Survival_SupplementalTable_S1_20171025_xena_sp` for cancer type and pathologic stage.
- **GDC clinical:** `data/gdc_clinical.tsv` for AJCC T/M/stage (pathologic preferred over clinical).


In [2]:
exp_df = pd.read_csv("data/tcga_gene_expected_count.gz", compression="gzip", sep="\t")
gene_map = pd.read_csv("data/probeMap_gencode.v23.annotation.gene.probemap", sep="\t")
clinical_df = pd.read_csv("data/Survival_SupplementalTable_S1_20171025_xena_sp", sep="\t")


## 3. Expression: gene symbols and primary tumor samples


In [3]:
exp_df["gene_symbol"] = exp_df["sample"].map(gene_map.set_index("id")["gene"])
reordered_cols = ["gene_symbol", "sample"] + [
    c for c in exp_df.columns if c not in ("gene_symbol", "sample")
]
exp_df = exp_df[reordered_cols]


In [4]:
primary_cols = ["gene_symbol", "sample"] + [
    c for c in exp_df.columns
    if c not in ("gene_symbol", "sample") and c.split("-")[3] == "01"
]
primary_exp_df = exp_df[primary_cols].copy()
# Drop Ensembl probe id column (still named "sample" in source)
primary_exp_df = primary_exp_df.rename(columns={"sample": "ensembl_id"}).drop(
    columns=["ensembl_id"]
)


## 4. Wide matrix: rows = aliquots, columns = gene symbols

Transpose so each row is a `sample_id` (TCGA barcode) and each column is a gene.


In [5]:
primary_exp_t = primary_exp_df.set_index("gene_symbol").T
primary_exp_t.index.name = None
primary_exp_t.columns.name = None
primary_exp_t = primary_exp_t.reset_index().rename(columns={"index": "sample_id"})


In [6]:
primary_exp_t["cancer_type"] = primary_exp_t["sample_id"].map(
    clinical_df.set_index("sample")["cancer type abbreviation"]
)
primary_exp_t["stage"] = primary_exp_t["sample_id"].map(
    clinical_df.set_index("sample")["ajcc_pathologic_tumor_stage"]
)


## 5. GDC clinical: one row per case, resolve conflicts

Aggregate duplicate `cases.submitter_id` rows, prefer pathologic AJCC over clinical, flag semicolon-joined conflicts, then drop conflicting cases. Simplified T and M columns feed the expression table.


In [8]:
clincal_gdc = pd.read_csv("data/gdc_clinical.tsv", sep="\t", low_memory=False)


In [9]:
clinical_cols = [
    "diagnoses.ajcc_clinical_m",
    "diagnoses.ajcc_clinical_stage",
    "diagnoses.ajcc_clinical_t",
    "diagnoses.ajcc_pathologic_m",
    "diagnoses.ajcc_pathologic_stage",
    "diagnoses.ajcc_pathologic_t",
]

clincal_gdc_sliced = clincal_gdc[["cases.submitter_id"] + clinical_cols].copy()

missing_like = ["", " ", "not reported", "Not Reported", "unknown", "Unknown", "--", "'--"]
clincal_gdc_sliced = clincal_gdc_sliced.replace(missing_like, np.nan)


In [10]:
def collapse_unique(x):
    vals = x.dropna().astype(str).unique()
    if len(vals) == 0:
        return np.nan
    if len(vals) == 1:
        return vals[0]
    return ";".join(vals)

clinical_one = (
    clincal_gdc_sliced.groupby("cases.submitter_id", as_index=False).agg(collapse_unique)
)


In [11]:
clinical_one["ajcc_m_best"] = clinical_one["diagnoses.ajcc_pathologic_m"].combine_first(
    clinical_one["diagnoses.ajcc_clinical_m"]
)
clinical_one["ajcc_t_best"] = clinical_one["diagnoses.ajcc_pathologic_t"].combine_first(
    clinical_one["diagnoses.ajcc_clinical_t"]
)
clinical_one["ajcc_stage_best"] = clinical_one[
    "diagnoses.ajcc_pathologic_stage"
].combine_first(clinical_one["diagnoses.ajcc_clinical_stage"])


In [12]:
clinical_one["stage_conflict"] = (
    clinical_one["diagnoses.ajcc_pathologic_stage"].astype(str).str.contains(";")
    | clinical_one["diagnoses.ajcc_clinical_stage"].astype(str).str.contains(";")
)
clinical_one["t_conflict"] = (
    clinical_one["diagnoses.ajcc_pathologic_t"].astype(str).str.contains(";")
    | clinical_one["diagnoses.ajcc_clinical_t"].astype(str).str.contains(";")
)
clinical_one["m_conflict"] = (
    clinical_one["diagnoses.ajcc_pathologic_m"].astype(str).str.contains(";")
    | clinical_one["diagnoses.ajcc_clinical_m"].astype(str).str.contains(";")
)


In [13]:
clinical_one_no_conflicts = clinical_one[
    ~clinical_one["stage_conflict"]
    & ~clinical_one["t_conflict"]
    & ~clinical_one["m_conflict"]
].copy()
clinical_one_no_conflicts.shape


(11098, 13)

In [14]:
def collapse_t_stage(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    if s.startswith("TIS"):
        return "Tis"
    if s.startswith("TX"):
        return "TX"
    if s.startswith("T0"):
        return "T0"
    m = re.match(r"^T([1-4])", s)
    if m:
        return f"T{m.group(1)}"
    return np.nan


def collapse_m_stage(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    m = re.search(r"M([01X])", s)
    if m:
        return f"M{m.group(1)}"
    return np.nan


clinical_one_no_conflicts["ajcc_t_simple"] = clinical_one_no_conflicts["ajcc_t_best"].apply(
    collapse_t_stage
)
clinical_one_no_conflicts["ajcc_m_simple"] = clinical_one_no_conflicts["ajcc_m_best"].apply(
    collapse_m_stage
)


In [ ]:
reduced_clin = clinical_one_no_conflicts[
    ["cases.submitter_id", "ajcc_t_simple", "ajcc_m_simple", "ajcc_stage_best"]
]
reduced_clin.to_csv("data/reduced_gdc_clinical_no_conflicts.csv", index=False)


## 6. Join GDC AJCC onto the expression table


In [16]:
primary_exp_t["patient_id"] = primary_exp_t["sample_id"].str.split("-").str[:3].str.join("-")
idx = reduced_clin.set_index("cases.submitter_id")
primary_exp_t["ajcc_m"] = primary_exp_t["patient_id"].map(idx["ajcc_m_simple"])
primary_exp_t["ajcc_t"] = primary_exp_t["patient_id"].map(idx["ajcc_t_simple"])
primary_exp_t["ajcc_stage"] = primary_exp_t["patient_id"].map(idx["ajcc_stage_best"])


In [17]:
first_cols = ["sample_id", "cancer_type", "stage", "ajcc_t", "ajcc_m", "ajcc_stage"]
cols = list(primary_exp_t.columns)
first_existing = [c for c in first_cols if c in cols]
first_pos = [cols.index(c) for c in first_existing]
first_set = set(first_existing)
rest_pos = [i for i, c in enumerate(cols) if c not in first_set]
primary_exp_t = primary_exp_t.iloc[:, first_pos + rest_pos]


In [18]:
primary_exp_t = primary_exp_t.copy()
primary_exp_t["stage"] = primary_exp_t["stage"].fillna(primary_exp_t["ajcc_stage"])


## 7. Export: samples with parsable stage (I–IV)

Writes `data/primary_exp_stage_final.csv` after coercing Xena/GDC stage strings to coarse Roman buckets.


In [ ]:
def collapse_ajcc_stage(x):
    if pd.isna(x):
        return np.nan
    s = str(x).strip().upper()
    m = re.match(r"^STAGE\s+(IV|III|II|I)(?:[A-C]?\d*)?$", s)
    if not m:
        return np.nan
    return m.group(1)


primary_exp_t["stage_general"] = primary_exp_t["stage"].apply(collapse_ajcc_stage)
primary_exp_final = primary_exp_t[primary_exp_t["stage_general"].notna()].copy()
primary_exp_final = primary_exp_final.drop(columns=["stage", "ajcc_stage"], errors="ignore")

first_cols = ["sample_id", "cancer_type", "stage_general", "ajcc_t", "ajcc_m"]
cols = list(primary_exp_final.columns)
first_pos = [i for i, c in enumerate(cols) if c in first_cols]
rest_pos = [i for i, c in enumerate(cols) if c not in first_cols]
primary_exp_final = primary_exp_final.iloc[:, first_pos + rest_pos].rename(
    columns={"stage_general": "stage"}
)
primary_exp_final.to_csv("data/primary_exp_stage_final.csv", index=False)
primary_exp_final.shape


(6256, 60504)

## 8. Export: metastasis-known subset (exclude `MX`)

Builds `M_stage` labels combining M0/M1 with T1 vs larger T. Saves `data/primary_exp_metastasis_final.csv`.


In [20]:
primary_exp_metastasis = primary_exp_t[primary_exp_t["ajcc_m"].notna()].copy()
primary_exp_metastasis = primary_exp_metastasis[
    primary_exp_metastasis["ajcc_m"] != "MX"
].copy()

primary_exp_metastasis["M_stage"] = pd.NA
m0 = primary_exp_metastasis["ajcc_m"].eq("M0")
m1 = primary_exp_metastasis["ajcc_m"].eq("M1")
t1 = primary_exp_metastasis["ajcc_t"].eq("T1")
t_known = primary_exp_metastasis["ajcc_t"].notna()

primary_exp_metastasis.loc[m0 & t1, "M_stage"] = "M0_s"
primary_exp_metastasis.loc[m0 & (~t1) & t_known, "M_stage"] = "M0_l"
primary_exp_metastasis.loc[m1 & t1, "M_stage"] = "M1_s"
primary_exp_metastasis.loc[m1 & (~t1) & t_known, "M_stage"] = "M1_L"


In [21]:
cols_to_drop = ["stage", "ajcc_t", "ajcc_m", "ajcc_stage"]
primary_exp_metastasis = primary_exp_metastasis.drop(columns=cols_to_drop, errors="ignore")

front_cols = ["sample_id", "cancer_type", "stage_general", "M_stage"]
for col in reversed(front_cols):
    if col in primary_exp_metastasis.columns:
        s = primary_exp_metastasis.pop(col)
        primary_exp_metastasis.insert(0, col, s)

primary_exp_metastasis = primary_exp_metastasis.rename(columns={"stage_general": "stage"})
# primary_exp_metastasis.to_csv("data/primary_exp_metastasis_final.csv", index=False)
primary_exp_metastasis.shape


(5339, 60503)

## 9. GENCODE v23: gene and lncRNA coordinates

Parse `data/gencode.v23.annotation.gtf.gz`, extract gene rows, save full gene table and a small lncRNA-related `gene_type` subset to CSV.


In [ ]:
gtf_path = "data/gencode.v23.annotation.gtf.gz"
gtf_cols = [
    "chrom", "source", "feature", "start", "end", "score", "strand", "frame", "attribute",
]
gtf = pd.read_csv(
    gtf_path,
    sep="\t",
    comment="#",
    names=gtf_cols,
    compression="gzip",
)
gtf.head()


In [ ]:
genes = gtf[gtf["feature"] == "gene"].copy()
genes["gene_id"] = genes["attribute"].str.extract(r'gene_id "([^"]+)"')
genes["gene_name"] = genes["attribute"].str.extract(r'gene_name "([^"]+)"')
genes["gene_type"] = genes["attribute"].str.extract(r'gene_type "([^"]+)"')
genes.to_csv("data/gencodev23_genes.csv", index=False)
genes[["chrom", "start", "end", "strand", "gene_id", "gene_name", "gene_type"]].head()


In [ ]:
lncrna_types = {
    "lincRNA",
    "antisense",
    "sense_intronic",
    "sense_overlapping",
    "processed_transcript",
    "3prime_overlapping_ncrna",
    "macro_lncRNA",
}
lncrna_genes = genes[genes["gene_type"].isin(lncrna_types)].copy()
lncrna_genes_small = lncrna_genes[
    ["chrom", "start", "end", "strand", "gene_id", "gene_name", "gene_type"]
].copy()
lncrna_genes_small.to_csv("data/lncrna_genes_small.csv", index=False)
lncrna_genes_small.shape


## 10. lncRNA-only expression matrices

Reload the saved wide tables (so this section can run standalone), keep metadata columns plus any column whose name is a GENCODE lncRNA `gene_name`, and write `data/primary_exp_stage_lncRNA.csv` and `data/primary_exp_metastasis_lncRNA.csv`.


In [ ]:
primary_exp_stage = pd.read_csv("data/primary_exp_stage_final.csv")
primary_exp_metastasis = pd.read_csv("data/primary_exp_metastasis_final.csv")
lncrna_genes_small = pd.read_csv("data/lncrna_genes_small.csv")


In [ ]:
metadata_cols = ["sample_id", "cancer_type", "ajcc_t", "ajcc_m", "stage"]
lncrna_symbols = set(lncrna_genes_small["gene_name"].dropna().astype(str))

keep_mask = [(c in metadata_cols) or (str(c) in lncrna_symbols) for c in primary_exp_stage.columns]
primary_exp_lncrna_stage = primary_exp_stage.loc[:, keep_mask].copy()
primary_exp_lncrna_stage.to_csv("data/primary_exp_stage_lncRNA.csv", index=False)
primary_exp_lncrna_stage.shape


In [ ]:
metadata_cols_m = ["sample_id", "cancer_type", "stage", "M_stage"]
keep_mask_m = [
    (c in metadata_cols_m) or (str(c) in lncrna_symbols)
    for c in primary_exp_metastasis.columns
]
primary_exp_lncrna_meta = primary_exp_metastasis.loc[:, keep_mask_m].copy()
primary_exp_lncrna_meta.to_csv("data/primary_exp_metastasis_lncRNA.csv", index=False)
primary_exp_lncrna_meta.shape
